Objective:
This project extracts hourly historical weather data for 50 major cities across India using the Open-Meteo Archive API.

The data covers the period from January 1, 2026 to August 20, 2026 and includes temperature, humidity, wind speed, visibility, atmospheric pressure, and weather conditions.

In [53]:
import requests
import pandas as pd
import time

cities = {
    "Delhi": (28.6139, 77.2090),
    "Mumbai": (19.0760, 72.8777),
    "Bengaluru": (12.9716, 77.5946),
    "Hyderabad": (17.3850, 78.4867),
    "Ahmedabad": (23.0225, 72.5714),
    "Chennai": (13.0827, 80.2707),
    "Kolkata": (22.5726, 88.3639),
    "Pune": (18.5204, 73.8567),
    "Jaipur": (26.9124, 75.7873),
    "Surat": (21.1702, 72.8311),
    "Lucknow": (26.8467, 80.9462),
    "Kanpur": (26.4499, 80.3319),
    "Nagpur": (21.1458, 79.0882),
    "Indore": (22.7196, 75.8577),
    "Bhopal": (23.2599, 77.4126),
    "Patna": (25.5941, 85.1376),
    "Vadodara": (22.3072, 73.1812),
    "Ludhiana": (30.9010, 75.8573),
    "Agra": (27.1767, 78.0081),
    "Nashik": (19.9975, 73.7898),
    "Faridabad": (28.4089, 77.3178),
    "Meerut": (28.9845, 77.7064),
    "Rajkot": (22.3039, 70.8022),
    "Varanasi": (25.3176, 82.9739),
    "Srinagar": (34.0837, 74.7973),
    "Aurangabad": (19.8762, 75.3433),
    "Dhanbad": (23.7957, 86.4304),
    "Amritsar": (31.6340, 74.8723),
    "Prayagraj": (25.4358, 81.8463),
    "Ranchi": (23.3441, 85.3096),
    "Howrah": (22.5958, 88.2636),
    "Coimbatore": (11.0168, 76.9558),
    "Jabalpur": (23.1815, 79.9864),
    "Gwalior": (26.2183, 78.1828),
    "Vijayawada": (16.5062, 80.6480),
    "Jodhpur": (26.2389, 73.0243),
    "Madurai": (9.9252, 78.1198),
    "Raipur": (21.2514, 81.6296),
    "Kota": (25.2138, 75.8648),
    "Chandigarh": (30.7333, 76.7794),
    "Guwahati": (26.1445, 91.7362),
    "Bhubaneswar": (20.2961, 85.8245),
    "Thiruvananthapuram": (8.5241, 76.9366),
    "Kochi": (9.9312, 76.2673),
    "Visakhapatnam": (17.6868, 83.2185),
    "Mysuru": (12.2958, 76.6394),
    "Dehradun": (30.3165, 78.0322),
    "Noida": (28.5355, 77.3910),
    "Gurugram": (28.4595, 77.0266),
    "Navi Mumbai": (19.0330, 73.0297)
}

url = "https://archive-api.open-meteo.com/v1/archive"

start_date = "2026-01-01"
end_date = "2026-08-20"

all_data = []

for city, (latitude, longitude) in cities.items():

    print(f"Fetching {city}...")

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": (
            "temperature_2m,"
            "dew_point_2m,"
            "relative_humidity_2m,"
            "wind_speed_10m,"
            "visibility,"
            "surface_pressure,"
            "weather_code"
        ),
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "timezone": "Asia/Kolkata"
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        hourly = data["hourly"]

        city_df = pd.DataFrame({
            "Date/Time": hourly["time"],
            "City": city,
            "Latitude": latitude,
            "Longitude": longitude,
            "Temp_C": hourly["temperature_2m"],
            "Dew Point Temp_C": hourly["dew_point_2m"],
            "Rel Hum_%": hourly["relative_humidity_2m"],
            "Wind Speed_km/h": hourly["wind_speed_10m"],
            "Visibility_km": [
                x / 1000 if x is not None else None
                for x in hourly["visibility"]
            ],
            "Press_kPa": [
                x / 10 if x is not None else None
                for x in hourly["surface_pressure"]
            ],
            "Weather": hourly["weather_code"]
        })

        all_data.append(city_df)

        print(f"   {len(city_df):,} rows")

    except Exception as e:

        print(f"   ERROR: {e}")

    time.sleep(1)


df = pd.concat(
    all_data,
    ignore_index=True
)

df["Date/Time"] = pd.to_datetime(df["Date/Time"])

df = df.sort_values(
    ["City", "Date/Time"]
).reset_index(drop=True)

print("\n==============================")
print("DATA EXTRACTION COMPLETED")
print("==============================")

print("Cities:", df["City"].nunique())
print("Rows:", f"{len(df):,}")

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))

df.to_csv(
    "India_Weather_2026_50_Cities.csv",
    index=False
)

print("\nCSV saved successfully!")

Fetching Delhi...
   5,568 rows
Fetching Mumbai...
   5,568 rows
Fetching Bengaluru...
   5,568 rows
Fetching Hyderabad...
   5,568 rows
Fetching Ahmedabad...
   5,568 rows
Fetching Chennai...
   5,568 rows
Fetching Kolkata...
   5,568 rows
Fetching Pune...
   5,568 rows
Fetching Jaipur...
   5,568 rows
Fetching Surat...
   5,568 rows
Fetching Lucknow...
   5,568 rows
Fetching Kanpur...
   5,568 rows
Fetching Nagpur...
   5,568 rows
Fetching Indore...
   5,568 rows
Fetching Bhopal...
   5,568 rows
Fetching Patna...
   5,568 rows
Fetching Vadodara...
   5,568 rows
Fetching Ludhiana...
   5,568 rows
Fetching Agra...
   5,568 rows
Fetching Nashik...
   5,568 rows
Fetching Faridabad...
   5,568 rows
Fetching Meerut...
   5,568 rows
Fetching Rajkot...
   5,568 rows
Fetching Varanasi...
   5,568 rows
Fetching Srinagar...
   5,568 rows
Fetching Aurangabad...
   5,568 rows
Fetching Dhanbad...
   5,568 rows
Fetching Amritsar...
   5,568 rows
Fetching Prayagraj...
   5,568 rows
Fetching Ranchi..

,Date/Time,City,Latitude,Longitude,Temp_C,Dew Point Temp_C,Rel Hum_%,Wind Speed_km/h,Visibility_km,Press_kPa,Weather
0,2026-01-01 00:00:00,Agra,27.1767,78.0081,11.2,10.4,95,9.2,None,99.28,3
1,2026-01-01 01:00:00,Agra,27.1767,78.0081,10.9,10.2,95,7.6,None,99.28,1
2,2026-01-01 02:00:00,Agra,27.1767,78.0081,10.5,9.9,96,7.4,None,99.29,0
3,2026-01-01 03:00:00,Agra,27.1767,78.0081,10.3,9.8,97,8.3,None,99.28,0
4,2026-01-01 04:00:00,Agra,27.1767,78.0081,10.0,9.7,98,8.0,None,99.27,0
5,2026-01-01 05:00:00,Agra,27.1767,78.0081,10.0,9.9,99,6.9,None,99.35,3
6,2026-01-01 06:00:00,Agra,27.1767,78.0081,10.2,10.1,100,4.2,None,99.41,3
7,2026-01-01 07:00:00,Agra,27.1767,78.0081,10.4,10.3,99,7.5,None,99.46,3
8,2026-01-01 08:00:00,Agra,27.1767,78.0081,10.9,10.5,97,7.7,None,99.54,3
9,2026-01-01 09:00:00,Agra,27.1767,78.0081,12.5,10.9,90,6.7,None,99.66,3



CSV saved successfully!


In [90]:
print(hourly["visibility"][:20])

[17940.0, 18020.0, 8920.0, 14740.0, 15560.0, 15600.0, 15760.0, 15760.0, 18040.0, 19520.0, 20780.0, 21860.0, 21860.0, 21860.0, 21860.0, 21860.0, 21880.0, 20800.0, 19540.0, 18060.0]


##Check Data

In [68]:
df.head()

,Date/Time,City,Latitude,Longitude,Temp_C,Dew Point Temp_C,Rel Hum_%,Wind Speed_km/h,Visibility_km,Press_kPa,Weather
0,2026-01-01 00:00:00,Agra,27.1767,78.0081,11.2,10.4,95,9.2,None,99.28,3
1,2026-01-01 01:00:00,Agra,27.1767,78.0081,10.9,10.2,95,7.6,None,99.28,1
2,2026-01-01 02:00:00,Agra,27.1767,78.0081,10.5,9.9,96,7.4,None,99.29,0
3,2026-01-01 03:00:00,Agra,27.1767,78.0081,10.3,9.8,97,8.3,None,99.28,0
4,2026-01-01 04:00:00,Agra,27.1767,78.0081,10.0,9.7,98,8.0,None,99.27,0


In [75]:
df.columns

Index(['Date/Time', 'City', 'Latitude', 'Longitude', 'Temp_C',
       'Dew Point Temp_C', 'Rel Hum_%', 'Wind Speed_km/h', 'Visibility_km',
       'Press_kPa', 'Weather'],
      dtype='str')

In [59]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 278400 entries, 0 to 278399
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Date/Time         278400 non-null  datetime64[us]
 1   City              278400 non-null  str           
 2   Latitude          278400 non-null  float64       
 3   Longitude         278400 non-null  float64       
 4   Temp_C            278400 non-null  float64       
 5   Dew Point Temp_C  278400 non-null  float64       
 6   Rel Hum_%         278400 non-null  int64         
 7   Wind Speed_km/h   278400 non-null  float64       
 8   Visibility_km     0 non-null       object        
 9   Press_kPa         278400 non-null  float64       
 10  Weather           278400 non-null  int64         
dtypes: datetime64[us](1), float64(6), int64(2), object(1), str(1)
memory usage: 23.4+ MB


In [64]:
import requests

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 20.2961,
    "longitude": 85.8245,
    "start_date": "2026-01-01",
    "end_date": "2026-01-03",
    "hourly": (
        "temperature_2m,"
        "dew_point_2m,"
        "relative_humidity_2m,"
        "wind_speed_10m,"
        "visibility,"
        "surface_pressure,"
        "weather_code"
    ),
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=120)

print(response.status_code)

data_test = response.json()

print(data_test["hourly"].keys())

visibility = data_test["hourly"]["visibility"]

print(visibility[:20])         

200
dict_keys(['time', 'temperature_2m', 'dew_point_2m', 'relative_humidity_2m', 'wind_speed_10m', 'visibility', 'surface_pressure', 'weather_code'])
[10160.0, 7100.0, 7100.0, 300.0, 160.0, 120.0, 120.0, 120.0, 360.0, 11780.0, 15060.0, 16280.0, 16280.0, 17360.0, 17360.0, 17360.0, 16300.0, 16300.0, 16300.0, 15100.0]


In [65]:
df_backup = df.copy()

In [70]:
import requests
import pandas as pd
import time

visibility_data = []

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

for city, (latitude, longitude) in cities.items():

    print(f"Fetching visibility: {city}...")

    params = {
        "latitude": latitude,
        "longitude": longitude,

        "start_date": "2026-01-01",
        "end_date": "2026-08-24",

        "hourly": "visibility",

        "timezone": "Asia/Kolkata"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        hourly = data["hourly"]

        temp_df = pd.DataFrame({
            "Date/Time": pd.to_datetime(hourly["time"]),
            "City": city,
            "Visibility_km": [
                x / 1000 if x is not None else None
                for x in hourly["visibility"]
            ]
        })

        visibility_data.append(temp_df)

        print(f"   ✓ {len(temp_df):,} rows")

    except Exception as e:
        print(f"   ❌ Error: {e}")

    time.sleep(1)


visibility_df = pd.concat(
    visibility_data,
    ignore_index=True
)

print("\nVisibility data:")
print(visibility_df.shape)

display(visibility_df.head())

Fetching visibility: Delhi...
   ✓ 5,664 rows
Fetching visibility: Mumbai...
   ✓ 5,664 rows
Fetching visibility: Bengaluru...
   ✓ 5,664 rows
Fetching visibility: Hyderabad...
   ✓ 5,664 rows
Fetching visibility: Ahmedabad...
   ✓ 5,664 rows
Fetching visibility: Chennai...
   ✓ 5,664 rows
Fetching visibility: Kolkata...
   ✓ 5,664 rows
Fetching visibility: Pune...
   ✓ 5,664 rows
Fetching visibility: Jaipur...
   ✓ 5,664 rows
Fetching visibility: Surat...
   ✓ 5,664 rows
Fetching visibility: Lucknow...
   ✓ 5,664 rows
Fetching visibility: Kanpur...
   ✓ 5,664 rows
Fetching visibility: Nagpur...
   ✓ 5,664 rows
Fetching visibility: Indore...
   ✓ 5,664 rows
Fetching visibility: Bhopal...
   ✓ 5,664 rows
Fetching visibility: Patna...
   ✓ 5,664 rows
Fetching visibility: Vadodara...
   ✓ 5,664 rows
Fetching visibility: Ludhiana...
   ✓ 5,664 rows
Fetching visibility: Agra...
   ✓ 5,664 rows
Fetching visibility: Nashik...
   ✓ 5,664 rows
Fetching visibility: Faridabad...
   ✓ 5,664 rows
F

,Date/Time,City,Visibility_km
0,2026-01-01 00:00:00,Delhi,7.16
1,2026-01-01 01:00:00,Delhi,7.16
2,2026-01-01 02:00:00,Delhi,7.16
3,2026-01-01 03:00:00,Delhi,7.16
4,2026-01-01 04:00:00,Delhi,5.86


In [93]:
##Final check
print(df.info())


print("\nMissing values:")
print(df.isnull().sum())


display(df.head(10))

<class 'pandas.DataFrame'>
RangeIndex: 278400 entries, 0 to 278399
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Date/Time         278400 non-null  datetime64[us]
 1   City              278400 non-null  str           
 2   Latitude          278400 non-null  float64       
 3   Longitude         278400 non-null  float64       
 4   Temp_C            278400 non-null  float64       
 5   Dew Point Temp_C  278400 non-null  float64       
 6   Rel Hum_%         278400 non-null  int64         
 7   Wind Speed_km/h   278400 non-null  float64       
 8   Visibility_km_x   0 non-null       float64       
 9   Press_kPa         278400 non-null  float64       
 10  Weather           278400 non-null  int64         
 11  Visibility_km_y   278400 non-null  float64       
dtypes: datetime64[us](1), float64(8), int64(2), str(1)
memory usage: 25.5 MB
None

Missing values:
Date/Time                0
Cit

,Date/Time,City,Latitude,Longitude,Temp_C,Dew Point Temp_C,Rel Hum_%,Wind Speed_km/h,Visibility_km_x,Press_kPa,Weather,Visibility_km_y
0,2026-01-01 00:00:00,Agra,27.1767,78.0081,11.2,10.4,95,9.2,NaN,99.28,3,7.70
1,2026-01-01 01:00:00,Agra,27.1767,78.0081,10.9,10.2,95,7.6,NaN,99.28,1,5.20
2,2026-01-01 02:00:00,Agra,27.1767,78.0081,10.5,9.9,96,7.4,NaN,99.29,0,5.20
3,2026-01-01 03:00:00,Agra,27.1767,78.0081,10.3,9.8,97,8.3,NaN,99.28,0,5.20
4,2026-01-01 04:00:00,Agra,27.1767,78.0081,10.0,9.7,98,8.0,NaN,99.27,0,5.20
5,2026-01-01 05:00:00,Agra,27.1767,78.0081,10.0,9.9,99,6.9,NaN,99.35,3,2.74
6,2026-01-01 06:00:00,Agra,27.1767,78.0081,10.2,10.1,100,4.2,NaN,99.41,3,0.48
7,2026-01-01 07:00:00,Agra,27.1767,78.0081,10.4,10.3,99,7.5,NaN,99.46,3,0.34
8,2026-01-01 08:00:00,Agra,27.1767,78.0081,10.9,10.5,97,7.7,NaN,99.54,3,5.14
9,2026-01-01 09:00:00,Agra,27.1767,78.0081,12.5,10.9,90,6.7,NaN,99.66,3,7.62


In [94]:
# Remove old empty visibility column
df = df.drop(columns=["Visibility_km_x"])

# Rename correct visibility column
df = df.rename(
    columns={"Visibility_km_y": "Visibility_km"}
)

# Check columns
print(df.columns.tolist())

['Date/Time', 'City', 'Latitude', 'Longitude', 'Temp_C', 'Dew Point Temp_C', 'Rel Hum_%', 'Wind Speed_km/h', 'Press_kPa', 'Weather', 'Visibility_km']


Final Data Quality Check

In [95]:
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 278400 entries, 0 to 278399
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Date/Time         278400 non-null  datetime64[us]
 1   City              278400 non-null  str           
 2   Latitude          278400 non-null  float64       
 3   Longitude         278400 non-null  float64       
 4   Temp_C            278400 non-null  float64       
 5   Dew Point Temp_C  278400 non-null  float64       
 6   Rel Hum_%         278400 non-null  int64         
 7   Wind Speed_km/h   278400 non-null  float64       
 8   Press_kPa         278400 non-null  float64       
 9   Weather           278400 non-null  int64         
 10  Visibility_km     278400 non-null  float64       
dtypes: datetime64[us](1), float64(7), int64(2), str(1)
memory usage: 23.4 MB
None

Missing values:
Date/Time           0
City                0
Latitude            0
Longitude          

In [96]:
##Final csv save 
df.to_csv(
    "India_Weather_2026_50_Cities_Final.csv",
    index=False
)

print("Final CSV saved successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Final CSV saved successfully!
Rows: 278400
Columns: 11


                Data Quality Check

In [97]:
df.isnull().sum()

Date/Time           0
City                0
Latitude            0
Longitude           0
Temp_C              0
Dew Point Temp_C    0
Rel Hum_%           0
Wind Speed_km/h     0
Press_kPa           0
Weather             0
Visibility_km       0
dtype: int64

In [98]:
##Duplicate check
duplicates = df.duplicated(
    subset=["City", "Date/Time"]
).sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


In [99]:
##Temperature validation
print("Minimum Temperature:", df["Temp_C"].min())
print("Maximum Temperature:", df["Temp_C"].max())

Minimum Temperature: -12.4
Maximum Temperature: 45.4


In [100]:
##Outliers identify 

df[
    (df["Temp_C"] < -20) |
    (df["Temp_C"] > 55)
]

,Date/Time,City,Latitude,Longitude,Temp_C,Dew Point Temp_C,Rel Hum_%,Wind Speed_km/h,Press_kPa,Weather,Visibility_km


In [35]:
##Humidity validation
df[
    (df["Rel Hum_%"] < 0) |
    (df["Rel Hum_%"] > 100)
]

,Date/Time,City,Latitude,Longitude,Temp_C,Dew Point Temp_C,Rel Hum_%,Wind Speed_km/h,Visibility_km,Press_kPa,Weather


In [36]:
##Wind validation
df[df["Wind Speed_km/h"] < 0]

,Date/Time,City,Latitude,Longitude,Temp_C,Dew Point Temp_C,Rel Hum_%,Wind Speed_km/h,Visibility_km,Press_kPa,Weather


In [39]:
##Visibility validation
print(df["Visibility_km"].describe().round(2))

count    278400.00
mean         18.57
std           8.85
min           0.06
25%          11.70
50%          19.28
75%          25.60
max          57.08
Name: Visibility_km, dtype: float64


In [101]:
##Pressure validation
print(df["Press_kPa"].describe().round(2))

count    278400.00
mean         97.76
std           3.15
min          83.11
25%          96.12
50%          98.46
75%         100.07
max         102.20
Name: Press_kPa, dtype: float64


In [43]:
##Weather code check
print(
    sorted(df["Weather"].unique())
)

[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(51), np.int64(53), np.int64(55), np.int64(61), np.int64(63), np.int64(65), np.int64(71), np.int64(73), np.int64(75)]


In [102]:
# Wheather_map  name column create

weather_map = {
    0: "Clear Sky",
    1: "Mainly Clear",
    2: "Partly Cloudy",
    3: "Overcast",
    51: "Light Drizzle",
    53: "Moderate Drizzle",
    55: "Dense Drizzle",
    61: "Slight Rain",
    63: "Moderate Rain",
    65: "Heavy Rain",
    71: "Slight Snow",
    73: "Moderate Snow",
    75: "Heavy Snow"
}

df["Weather_Name"] = df["Weather"].map(weather_map)


In [103]:
##check
df[["Weather", "Weather_Name"]].drop_duplicates().sort_values("Weather")

,Weather,Weather_Name
2,0,Clear Sky
1,1,Mainly Clear
23,2,Partly Cloudy
0,3,Overcast
16,51,Light Drizzle
14,53,Moderate Drizzle
644,55,Dense Drizzle
751,61,Slight Rain
635,63,Moderate Rain
642,65,Heavy Rain


In [47]:
print(df["Weather_Name"].isnull().sum())

0


In [48]:
##Final CSV save
df.to_csv(
    "India_Weather_2026_50_Cities_Final.csv",
    index=False
)


print("CSV saved successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

CSV saved successfully!
Rows: 278400
Columns: 12


##Final Verificatio

In [77]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 278400 entries, 0 to 278399
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Date/Time         278400 non-null  datetime64[us]
 1   City              278400 non-null  str           
 2   Latitude          278400 non-null  float64       
 3   Longitude         278400 non-null  float64       
 4   Temp_C            278400 non-null  float64       
 5   Dew Point Temp_C  278400 non-null  float64       
 6   Rel Hum_%         278400 non-null  int64         
 7   Wind Speed_km/h   278400 non-null  float64       
 8   Visibility_km     0 non-null       object        
 9   Press_kPa         278400 non-null  float64       
 10  Weather           278400 non-null  int64         
dtypes: datetime64[us](1), float64(6), int64(2), object(1), str(1)
memory usage: 23.4+ MB


In [104]:
df.head()

,Date/Time,City,Latitude,Longitude,Temp_C,Dew Point Temp_C,Rel Hum_%,Wind Speed_km/h,Press_kPa,Weather,Visibility_km,Weather_Name
0,2026-01-01 00:00:00,Agra,27.1767,78.0081,11.2,10.4,95,9.2,99.28,3,7.7,Overcast
1,2026-01-01 01:00:00,Agra,27.1767,78.0081,10.9,10.2,95,7.6,99.28,1,5.2,Mainly Clear
2,2026-01-01 02:00:00,Agra,27.1767,78.0081,10.5,9.9,96,7.4,99.29,0,5.2,Clear Sky
3,2026-01-01 03:00:00,Agra,27.1767,78.0081,10.3,9.8,97,8.3,99.28,0,5.2,Clear Sky
4,2026-01-01 04:00:00,Agra,27.1767,78.0081,10.0,9.7,98,8.0,99.27,0,5.2,Clear Sky
